In [ ]:
import os
print("Current working directory:", os.getcwd())

In [ ]:
import sys
import os
repo_root = "/Users/nullset/Developer/ML/learning-project"

print("Repo root:", repo_root)
if repo_root not in sys.path:
    sys.path.append(repo_root)

from src import features  # noqa: E402


In [ ]:
raw_path = os.path.join(repo_root, "data", "raw")
interim_path = os.path.join(repo_root, "data", "interim")

features.process_folder(raw_path, interim_path, augment=True)

In [ ]:
import os
import shutil
import random

processed_path = os.path.join(repo_root, "data", "processed")

def split_dataset(interim_dir, processed_dir, train_ratio=0.7, val_ratio=0.15):
    os.makedirs(processed_dir, exist_ok=True)
    for split in ["train", "val", "test"]:
        os.makedirs(os.path.join(processed_dir, split), exist_ok=True)

    for class_name in os.listdir(interim_dir):
        class_path = os.path.join(interim_dir, class_name)
        if not os.path.isdir(class_path):
            continue

        files = [
            f for f in os.listdir(class_path)
            if os.path.isfile(os.path.join(class_path, f)) and f.lower().endswith(".npy")
        ]
        random.shuffle(files)

        n = len(files)
        if n == 0:
            continue

        n_train = int(train_ratio * n)
        n_val = int(val_ratio * n)

        splits = {
            "train": files[:n_train],
            "val": files[n_train:n_train+n_val],
            "test": files[n_train+n_val:]
        }

        for split, split_files in splits.items():
            split_class_dir = os.path.join(processed_dir, split, class_name)
            os.makedirs(split_class_dir, exist_ok=True)
            for f in split_files:
                src = os.path.join(class_path, f)
                dst = os.path.join(split_class_dir, f)
                shutil.copy(src, dst)

split_dataset(interim_path, processed_path)
print("Dataset split complete")

In [ ]:
import numpy as np

def load_npy_dataset(folder):
    X = []
    y = []

    for class_name in os.listdir(folder):
        class_path = os.path.join(folder, class_name)
        if not os.path.isdir(class_path):
            continue

        for file in os.listdir(class_path):
            if file.endswith(".npy"):
                X.append(np.load(os.path.join(class_path, file)))
                y.append(int(class_name))

    return np.array(X), np.array(y)

X_train, y_train = load_npy_dataset(os.path.join(processed_path, "train"))
X_val, y_val = load_npy_dataset(os.path.join(processed_path, "val"))

In [ ]:
from tensorflow.keras import layers, models # type: ignore

model = models.Sequential([
    layers.Input(shape=(64,64,3)),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(3, activation='softmax')
])

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=10)

In [ ]:
model.save(os.path.join(repo_root, "models", "hand_sign_cnn.h5"))

In [ ]:
X_test, y_test = load_npy_dataset(os.path.join(processed_path, "test"))

model.evaluate(X_test, y_test)

In [ ]:
sample = X_test[5]
pred = model.predict(sample.reshape(1,64,64,3))

print("Predicted:", np.argmax(pred))
print("Actual:", y_test[5])